# Analyze Attribution Outputs

This notebook is a guided viewer for exported attribution results, with a focus on TrackStar outputs.

It is meant to answer practical questions such as:

- What files did the run produce?
- Which EWoK items were hardest for the model?
- Which BOS rows show up repeatedly across many target items?
- Which rows look globally influential versus domain-specific?
- How do I inspect one target or one row without reading raw JSONL files by hand?

The notebook defaults to the successful `trackstar_step16000` folder you just produced, but you can point `OUTPUT_DIR` at any attribution output directory with the same export layout.

## What The Files Mean

The notebook is built around the shared attribution export format:

- `top_rows_stepXXXXXXXX.csv`
  Top-ranked candidate rows for each target item.
- `bottom_rows_stepXXXXXXXX.csv`
  Lowest-ranked candidate rows for each target item when `--bottomk` was enabled.
- `row_summary_stepXXXXXXXX.csv`
  Candidate rows aggregated across all target items at one checkpoint.
- `domain_summary_stepXXXXXXXX.csv`
  Candidate rows aggregated separately within each EWoK domain.
- `target_diagnostics_stepXXXXXXXX.jsonl`
  Per-target scalar diagnostics such as `s11`, `s12`, margins, and softplus loss.
- `target_items.jsonl`
  The actual EWoK item text and metadata.
- `checkpoint_compare.csv`
  Cross-checkpoint comparison table. This is empty for a single-checkpoint run.
- `run_summary.json`
  Provenance and config summary for the run.

You do **not** need to understand TrackStar internals to use this notebook. The main working idea is simple:

- targets are EWoK items
- candidate rows are BOS training rows
- the output scores tell you which candidate rows look most aligned with the query gradient for each target

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Markdown, display

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None


def require_pyplot():
    if plt is None:
        raise ImportError(
            "matplotlib is required for plotting cells in this notebook. "
            "Table-oriented cells still work without it."
        )
    return plt


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "research").is_dir() and (candidate / "tests").exists():
            return candidate
    fallback = Path("/home/jorge/tokenPred/moonshotGPT")
    if fallback.exists():
        return fallback
    raise FileNotFoundError("Could not locate the moonshotGPT repo root from the notebook working directory.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Most of the notebook's loading and summary behavior lives in
# analysis/attribution/common/notebook_analysis.py.
from research.bos_aligned_proto.analysis.attribution.common.notebook_analysis import (
    artifact_inventory,
    attach_row_text,
    load_attribution_run,
    merge_target_metadata,
    most_recurrent_rows,
    plot_checkpoint_compare,
    plot_domain_heatmap,
    plot_row_score_distribution,
    plot_target_loss_distribution,
    row_report,
    run_overview,
    step_overview,
    target_report,
    top_candidate_rows,
    top_domain_rows,
    top_targets_by_loss,
)

if plt is not None:
    plt.style.use("seaborn-v0_8-whitegrid")
REPO_ROOT

In [ ]:
DEFAULT_OUTPUT_DIR = Path(
    "/home/jorge/tokenPred/moonshotGPT/runs/research/bos_aligned_proto/"
    "babygpt_fineweb_bosrow_mbs4_T1024_d1024_h16_L24_tok491520_efftok491520_ws8_gas15_seed42_steps30000/"
    "analysis/attribution/trackstar_step16000"
)

# This bundled smoke example only covers a subset of EWoK targets/domains.
# Later cells derive their default domain from the loaded artifacts.
# Change only this line when you want to analyze a different attribution folder.
OUTPUT_DIR = DEFAULT_OUTPUT_DIR

run = load_attribution_run(OUTPUT_DIR)
display(Markdown(f"## Loaded Output Directory\n`{run.output_dir}`"))
display(run_overview(run))

In [ ]:
display(Markdown("## File Inventory"))
display(artifact_inventory(run))

display(Markdown("## Per-Step Summary"))
step_table = pd.concat([step_overview(run, step) for step in run.available_steps], ignore_index=True)
display(step_table)

In [ ]:
STEP = run.default_step

top_rows = run.load_top_rows(STEP)
row_summary = run.load_row_summary(STEP)
domain_summary = run.load_domain_summary(STEP)
diagnostics = run.load_target_diagnostics(STEP)
target_items = run.load_target_items()

print(f"Analyzing step {STEP}")
print(f"top_rows shape: {top_rows.shape}")
print(f"row_summary shape: {row_summary.shape}")
print(f"domain_summary shape: {domain_summary.shape}")
print(f"diagnostics shape: {diagnostics.shape}")
print(f"target_items shape: {target_items.shape}")

## Quick Checks

These are the first tables I usually inspect:

- hardest targets by `softplus_loss`
- most influential rows overall by `mean_abs_score`
- rows that recur in many target-level top-k lists

Interpretation tip:

- `softplus_loss` high: the model is struggling more on that EWoK item
- `mean_abs_score` high: the row has strong attribution magnitude overall
- many `target_hits`: the same row is showing up repeatedly across targets

In [ ]:
hard_targets = merge_target_metadata(top_targets_by_loss(diagnostics, top_n=20), target_items)
overall_rows = attach_row_text(top_candidate_rows(row_summary, top_n=20, by="mean_abs_score"), run, max_chars=220)
negative_rows = attach_row_text(top_candidate_rows(row_summary, top_n=20, by="negative_score_sum", ascending=True), run, max_chars=220)
recurrent_rows = attach_row_text(most_recurrent_rows(top_rows, top_n=20), run, max_chars=220)

display(Markdown("### Hardest Targets By Softplus Loss"))
display(hard_targets)

display(Markdown("### Most Influential Rows Overall"))
display(overall_rows)

display(Markdown("### Most Negative Rows Overall"))
display(negative_rows)

display(Markdown("### Rows That Recur Across Many Targets"))
display(recurrent_rows)

In [ ]:
plt = require_pyplot()
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_target_loss_distribution(diagnostics, ax=axes[0])
plot_row_score_distribution(row_summary, ax=axes[1], column="mean_abs_score")
plt.tight_layout()

## Domain Specialization Check

This heatmap starts from the most influential overall rows, then shows how strong each row looks inside each EWoK domain.

Use it to separate:

- broad rows that matter across many domains
- narrow rows that matter mostly for one domain

In [ ]:
plt = require_pyplot()
fig, ax = plt.subplots(figsize=(10, 6))
_, heatmap_table = plot_domain_heatmap(domain_summary, row_summary, top_n_rows=12, ax=ax)
plt.tight_layout()
display(heatmap_table)

## Inspect One Target

Set `TARGET_ID` to any target you care about. By default, the notebook picks the highest-loss target from the table above.

The report shows a deliberately small view:

- `pair_1`
- `pair_2`
- `context_type`
- `context_diff`
- `concept`
- top positive influence rows with just `softplus_loss`, `influence_direction`, `row_idx`, and `row_text`

In [ ]:
TARGET_ID = hard_targets["target_id"].iloc[0] if not hard_targets.empty else None

if TARGET_ID is None:
    print("No targets were available.")
else:
    target_views = target_report(
        run,
        top_rows,
        diagnostics,
        target_items,
        TARGET_ID,
        step=STEP,
        row_summary=row_summary,
        top_n_rows=20,
        bottom_n_rows=20,
        row_text_chars=320,
    )
    display(Markdown(f"### Target: `{TARGET_ID}`"))
    display(target_views["target_item"].T)
    display(Markdown("#### Top Positive Rows"))
    display(target_views["top_positive_rows"])


## Inspect One Candidate Row

Set `ROW_ID` to any BOS row you want to inspect. By default, the notebook picks the strongest row by overall `mean_abs_score`.

The report shows:

- the row's decoded BOS row text and overall summary across all targets
- the row's per-domain summary
- the targets where that row appears in the top-k list, with `C1/T1` and `C2/T2` text

In [ ]:
ROW_ID = int(overall_rows.iloc[0]["row_id"]) if not overall_rows.empty else None

if ROW_ID is None:
    print("No candidate rows were available.")
else:
    row_views = row_report(run, ROW_ID, top_rows, row_summary, domain_summary, top_n_targets=15)
    display(Markdown(f"### Row: `{ROW_ID}`"))
    display(row_views["overall_summary"].T)
    display(row_views["domain_summary"])
    display(row_views["top_target_appearances"])


## Domain-Specific Leaderboard

If you want to focus on one EWoK domain, set `DOMAIN_NAME` below. You can pass either the raw domain name such as `"material-dynamics"` or the exported group name such as `"domain:material-dynamics"`.

The default now picks the first exported domain in the loaded folder so the bundled smoke example works without manual edits.

In [ ]:
available_domain_groups = sorted(
    group
    for group in domain_summary.get("group", pd.Series(dtype=str)).dropna().unique()
    if str(group).startswith("domain:")
)
available_domains = [group.removeprefix("domain:") for group in available_domain_groups]

DOMAIN_NAME = available_domains[0] if available_domains else None

if DOMAIN_NAME is None:
    print("No domain-level rows were available in this output directory.")
else:
    display(Markdown(f"Using domain `{DOMAIN_NAME}`"))
    display(top_domain_rows(domain_summary, domain=DOMAIN_NAME, top_n=15, by="mean_abs_score"))

## Checkpoint Compare

This section only becomes useful once the output directory contains more than one checkpoint step.

For a single-checkpoint run, `checkpoint_compare.csv` is expected to be empty.

In [ ]:
checkpoint_compare = run.load_checkpoint_compare()

if checkpoint_compare.empty:
    print("checkpoint_compare.csv is empty. That is expected when only one checkpoint was scored.")
else:
    plt = require_pyplot()
    display(checkpoint_compare)
    fig, ax = plt.subplots(figsize=(8, 4))
    plot_checkpoint_compare(checkpoint_compare, ax=ax)
    plt.tight_layout()

## Suggested Next Uses

Once this notebook is working on one folder, the natural next steps are:

1. Point `OUTPUT_DIR` at a larger TrackStar run with the full EWoK bundle.
2. Point `OUTPUT_DIR` at a multi-checkpoint TrackStar run so `checkpoint_compare.csv` becomes informative.
3. Run a matched TRAK output folder and compare the same summary tables manually.
4. If you enabled dense scores, add target-by-row clustering or row-by-domain clustering on top of the exported matrix.